In [1]:
# Imports
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision
import os
import time
import datetime
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import json
from Codigo_graficas.graficas import graficar_confianza_vs_acierto, graficar_accuracy_por_clase
from Codigo_graficas.m_confusion import generar_matriz_confusion

2026-04-08 15:48:21.291822: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 15:48:21.318092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# CONFIGURACIÓN
ruta_test   = "./Testing"
ruta_output = "./Estudios"
 
IMG_SIZE    = 300       
BATCH_SIZE  = 64
NUM_CLASSES = 11
EPOCHS      = 20
nombre_archivo = "EffectiveNet_combinacion2"

# EVALUACIÓN CON EL MEJOR MODELO
print("\n" + "="*60)
print("📊 EVALUACIÓN FINAL")
print("="*60)



📊 EVALUACIÓN FINAL


In [3]:
# 1. CARGAR DATOS DE TEST
AUTOTUNE = tf.data.AUTOTUNE
test_datagen = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,  # Importante para mantener orden
    labels='inferred'
)

# Obtener nombres de clases
class_names = test_datagen.class_names
print(f"\n📋 Clases encontradas: {class_names}")
print(f"Total clases: {len(class_names)}")

# Preprocesar
test_datagen = (
    test_datagen
    .map(lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

Found 1379716 files belonging to 12 classes.
Using 1241745 files for training.


I0000 00:00:1774909273.173398  943336 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1176 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 1379716 files belonging to 12 classes.
Using 137971 files for validation.
Found 1088666 files belonging to 12 classes.

✅ Datasets listos 🚀


In [5]:
# 2. CARGAR MODELO
ruta_mejor_modelo = os.path.join(ruta_output, "Modelo", 
                                  "mejor_modelo_efficientnetb3_combinacion2_ft.keras")
model = load_model(ruta_mejor_modelo)
print(f"\n✅ Modelo cargado desde: {ruta_mejor_modelo}")


Resumen del modelo:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 300, 300,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 300, 300,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 300, 300,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 301, 301,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 150, 150,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 150, 150,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 150, 150,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 150, 150,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 150, 150,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 150, 150,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 150, 150,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 150, 150,  │        960 │ block1a_se_excit

 Total params: 10,981,819 (41.89 MB)

 Trainable params: 198,284 (774.55 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

In [6]:
# 3. EVALUACIÓN BÁSICA
print("\n" + "="*60)
print("📈 EVALUACIÓN GLOBAL")
print("="*60)

loss, acc = model.evaluate(test_datagen, verbose=1)
print(f"\n🎯 Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"📉 Test Loss: {loss:.4f}")

✅ Callbacks configurados


In [ ]:
# 4. MATRIZ DE CONFUSIÓN
print("\n" + "="*60)
print("🎨 VISUALIZANDO MATRIZ DE CONFUSIÓN")
print("="*60)


#creamos un nuevo datagen
test_datagen_fresh = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    labels='inferred'
)
test_datagen_fresh = (
    test_datagen_fresh
    .map(lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

n = nombre_archivo + "_confusion_matrix_test.png"
resultados_matriz = generar_matriz_confusion(
    model=model,
    test_datagen=test_datagen_fresh,
    class_names=class_names,
    guardar=True,
    ruta_guardado=os.path.join(ruta_output, "Matriz confusion", n),
    mostrar_reporte=True
)


🎯 FASE 1: ENTRENANDO CABEZA DE CLASIFICACIÓN
Epoch 1/20


2026-03-31 00:26:49.264389: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:20: Filling up shuffle buffer (this may take a while): 40 of 1000
2026-03-31 00:26:59.387724: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:20: Filling up shuffle buffer (this may take a while): 97 of 1000
2026-03-31 00:27:09.545126: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:20: Filling up shuffle buffer (this may take a while): 133 of 1000


In [ ]:
# Extraer resultados para usar después
y_true = resultados_matriz['y_true']
y_pred = resultados_matriz['y_pred']
cm = resultados_matriz['confusion_matrix']
acc = resultados_matriz['accuracy'] 

In [ ]:
# 5. MÉTRICAS ADICIONALES (opcional, para multiclase)
print("\n" + "="*60)
print("📊 MÉTRICAS AVANZADAS")
print("="*60)

# Exactitud por clase
class_accuracy = cm.diagonal() / cm.sum(axis=1)
for i, cls in enumerate(class_names):
    print(f"{cls:20s}: {class_accuracy[i]:.4f} ({class_accuracy[i]*100:.2f}%)")

# Error por clase
class_error = 1 - class_accuracy
worst_classes = np.argsort(class_error)[-5:][::-1]
print(f"\nPeores 5 clases:")
for idx in worst_classes:
    print(f"  - {class_names[idx]}: error {class_error[idx]:.4f}")

In [ ]:
# 6. ANÁLISIS DE CONFIANZA
print("\n" + "="*60)
print("🎯 ANÁLISIS DE CONFIANZA DE PREDICCIONES")
print("="*60)

print("🔄 Recalculando probabilidades para análisis de confianza...")
y_pred_probs = model.predict(test_datagen_fresh, verbose=0)
confianzas = np.max(y_pred_probs, axis=1)

print(f"📊 Estadísticas de confianza:")
print(f"  - Media: {np.mean(confianzas):.4f}")
print(f"  - Mediana: {np.median(confianzas):.4f}")
print(f"  - Mínima: {np.min(confianzas):.4f}")
print(f"  - Máxima: {np.max(confianzas):.4f}")
print(f"  - Desviación: {np.std(confianzas):.4f}")

# Predicciones con baja confianza
baja_confianza = np.where(confianzas < 0.6)[0]
print(f"\n⚠️ Predicciones con confianza < 0.6: {len(baja_confianza)} ({len(baja_confianza)/len(confianzas)*100:.2f}%)")

# Análisis de confianza por clase
print(f"\n📊 Confianza promedio por clase:")
for i, cls in enumerate(class_names):
    mask = y_pred == i
    if np.any(mask):
        conf_media_clase = np.mean(confianzas[mask])
        print(f"  {cls:25s}: {conf_media_clase:.4f}")
    else:
        print(f"  {cls:25s}: Sin predicciones")

In [ ]:
# 9. GUARDAR RESULTADOS COMPLETOS
print("\n" + "="*60)
print("GUARDANDO RESULTADOS")
print("="*60)

# Guardar predicciones
results_df = pd.DataFrame({
    'true_label': y_true,
    'true_class': [class_names[i] for i in y_true],
    'pred_label': y_pred,
    'pred_class': [class_names[i] for i in y_pred],
    'confidence': confianzas,
    'correct': y_true == y_pred
})
n = nombre_archivo + "_test_predictions.csv"
results_df.to_csv(os.path.join(ruta_output, "Entrenamiento", n), index=False)



# Guardar métricas resumidas
metrics = {
    'test_accuracy': float(acc),
    'test_loss': float(loss),
    'total_samples': len(y_true),
    'correct_predictions': int(np.sum(y_true == y_pred)),
    'mean_confidence': float(np.mean(confianzas)),
    'num_classes': NUM_CLASSES
}
n = nombre_archivo + "_test_metrics.json"
with open(os.path.join(ruta_output, "Metricas", n), 'w') as f:
    json.dump(metrics, f, indent=4)

print("Resultados guardados exitosamente")

In [ ]:
# 10. Grafica de accuracy por clase
n = nombre_archivo + "_accuracy_por_clase_test.png" 
graficar_accuracy_por_clase(
    cm=cm,
    class_names=class_names,
    titulo="Accuracy por clase - Test",
    guardar=True,
    ruta_guardado=os.path.join(ruta_output, "Graficas", n)
)

In [ ]:
# 11. Confianza vs acierto
n = nombre_archivo + "_confianza_vs_acierto_test.png" 
graficar_confianza_vs_acierto(
    confianzas=confianzas,
    y_true=y_true,
    y_pred=y_pred,
    titulo="Confianza vs acierto - Test",
    guardar=True,
    ruta_guardado=os.path.join(ruta_output, "Graficas", n)
)

In [ ]:
# 12. RESUMEN FINAL
print("\n" + "="*60)
print("RESUMEN FINAL DEL MODELO")
print("="*60)
print(f"🎯 Accuracy en test: {acc:.4f} ({acc*100:.2f}%)")
print(f"✅ Predicciones correctas: {np.sum(y_true == y_pred)}/{len(y_true)}")
print(f"❌ Predicciones incorrectas: {np.sum(y_true != y_pred)}/{len(y_true)}")
print(f"🎲 Confianza promedio: {np.mean(confianzas):.4f}")
print("="*60)
